In [1]:
import os
import sys

import numpy as np
import scipy

In [2]:
sys.path.insert(0, '../OptimalNumberOfTopics')

In [3]:
import topnum

from topnum.scores.perplexity_score import PerplexityScore
from topnum.scores.diversity_score import DiversityScore, KNOWN_METRICS
from topnum.model_constructor import init_model_from_family, KnownModel, PARAMS_EXPLORED

In [4]:
import topicnet
from topicnet.cooking_machine.dataset import Dataset
from topicnet.cooking_machine.models import (
    BaseScore as BaseTopicNetScore,
    TopicModel
)
from topicnet.cooking_machine.models.thetaless_regularizer import (
    dataset2sparse_matrix,
)

In [5]:
topicnet.__file__

! ls /home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/dataset_manager/

20NG.csv	 MKB10__internals	    RTL_Wiki.csv
20NG__internals  postnauka.csv		    RTL_Wiki_person.csv
api.py		 postnauka__internals	    RTL_Wiki_person__internals
Brown		 postnauka_noow.csv	    ruwiki_good__internals
Brown_BOW.csv	 postnauka_noow__internals  ruwiki_good.txt
Brown_NOOW.csv	 __pycache__		    WikiRef-220
hf		 Reuters		    wiki_ref220_bow.csv
__init__.py	 Reuters_BOW.csv	    wiki_ref220_natural_order.csv
MKB10.csv	 Reuters_NOOW.csv


In [6]:
DATA_FOLDER_PATH = '/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/dataset_manager'

In [7]:
! ls $DATA_FOLDER_PATH

20NG.csv	 MKB10__internals	    RTL_Wiki.csv
20NG__internals  postnauka.csv		    RTL_Wiki_person.csv
api.py		 postnauka__internals	    RTL_Wiki_person__internals
Brown		 postnauka_noow.csv	    ruwiki_good__internals
Brown_BOW.csv	 postnauka_noow__internals  ruwiki_good.txt
Brown_NOOW.csv	 __pycache__		    WikiRef-220
hf		 Reuters		    wiki_ref220_bow.csv
__init__.py	 Reuters_BOW.csv	    wiki_ref220_natural_order.csv
MKB10.csv	 Reuters_NOOW.csv


In [8]:
dataset = Dataset(
    f'{DATA_FOLDER_PATH}/ruwiki_good.txt',
)

dataset.get_possible_modalities()

{'@categories', '@lemmatized', '@ngramms'}

In [9]:
dataset._internals_folder_path

'/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/dataset_manager/ruwiki_good__internals'

In [10]:
! ls '/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/dataset_manager/'

20NG.csv	 MKB10__internals	    RTL_Wiki.csv
20NG__internals  postnauka.csv		    RTL_Wiki_person.csv
api.py		 postnauka__internals	    RTL_Wiki_person__internals
Brown		 postnauka_noow.csv	    ruwiki_good__internals
Brown_BOW.csv	 postnauka_noow__internals  ruwiki_good.txt
Brown_NOOW.csv	 __pycache__		    WikiRef-220
hf		 Reuters		    wiki_ref220_bow.csv
__init__.py	 Reuters_BOW.csv	    wiki_ref220_natural_order.csv
MKB10.csv	 Reuters_NOOW.csv


In [11]:
MAIN_MODALITY = '@lemmatized'

In [12]:
dataset._data.head()

,vw_text,raw_text,id
id,,,
Санкт-Петербург,Санкт-Петербург |@lemmatized год:301 петроград...,,Санкт-Петербург
Дворцовая_площадь,Дворцовая_площадь |@lemmatized дворцовый:43 пл...,,Дворцовая_площадь
Греко-персидские_войны,Греко-персидские_войны |@lemmatized грёкий:23 ...,,Греко-персидские_войны
Тихий_океан,Тихий_океан |@lemmatized тихий:92 океан:174 ус...,,Тихий_океан
Атлантический_океан,Атлантический_океан |@lemmatized атлантический...,,Атлантический_океан


In [13]:
dataset._data.shape

(8603, 3)

In [14]:
def calc_doc_occurrences(dataset, modality):
    """
    :param n_dw_matrix: sparse document-word matrix, shape is D x W
    :return: sparse matrix of co-occurrences

    doc_occurrences[w1, w2] = the number of the documents
    where there are w1 and w2
    """
    n_dw_matrix = dataset2sparse_matrix(dataset, modality, modalities_to_use=[modality])
    matrix = (scipy.sparse.csc_matrix(n_dw_matrix) > 0).astype(int)
    co_occurrences = matrix.T * matrix

    return co_occurrences.diagonal(), co_occurrences


def create_pmi_top_function(
    doc_occurrences, doc_co_occurrences,
    documents_number, top_sizes,
    topic_indices,
    co_occurrences_smooth=1.
):
    """
    :param doc_occurrences: array of doc occurrences of words
    :param doc_co_occurrences: sparse matrix of doc co-occurrences of words
    :param documents_number: number of the documents
    :param top_sizes: list of top values to calculate top-pmi for
    :param co_occurrences_smooth: constant to smooth co-occurrences in log
    :return: function which takes phi and theta and returns
    pair of two arrays: pmi-s of the tops and ppmi-s of the tops

    pmi[i] - pmi(top of size top_sizes[i])
    ppmi[i] - ppmi(top of size top_sizes[i])

    pmi(words) = sum_{u in words, v in words, u != v}
    log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    )

    ppmi(words) = sum_{u in words, v in words, u != v}
    max(log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    ), 0)

    """
    def func(phi):
        _T, W = phi.shape
        T = len(topic_indices)

        max_top_size = max(top_sizes)
        topic_pmis, topic_ppmis = dict(), dict()
        pmi, ppmi = np.zeros(max_top_size), np.zeros(max_top_size)
        tops = np.argpartition(phi, -max_top_size, axis=1)[:, -max_top_size:]
        
        for t in topic_indices:
            top = sorted(tops[t], key=lambda w: - phi[t, w])
            co_occurrences = doc_co_occurrences[top, :][:, top].todense()
            occurrences = doc_occurrences[top]
            values = np.log(
                (co_occurrences * documents_number + co_occurrences_smooth)
                / (occurrences[:, np.newaxis] * occurrences[np.newaxis, :] + co_occurrences_smooth)
            )
            diag = np.diag_indices(len(values))
            # values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()

            current_pmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_pmis[t] = current_pmi
            pmi += current_pmi

            values[values < 0.] = 0.
            current_ppmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_ppmis[t] = current_ppmi
            ppmi += current_ppmi
            
        sizes = np.arange(2, max_top_size + 1)
        pmi[1:] /= (T * sizes * (sizes - 1))
        ppmi[1:] /= (T * sizes * (sizes - 1))
        indices = np.array(top_sizes) - 1

        for t in topic_indices:
            topic_pmis[t][1:] /= (sizes * (sizes - 1))
            topic_ppmis[t][1:] /= (sizes * (sizes - 1))

        result_topic_pmis = {t: p[indices] for t, p in topic_pmis.items()}
        result_topic_ppmis = {t: p[indices] for t, p in topic_ppmis.items()}

        return pmi[indices], ppmi[indices], result_topic_pmis, result_topic_ppmis

    return func

In [163]:
%%time

occurences, co_occurences = calc_doc_occurrences(dataset, MAIN_MODALITY)

CPU times: user 1min 31s, sys: 32.1 s, total: 2min 3s
Wall time: 1min 59s


In [164]:
co_occurences.shape

(265247, 265247)

In [15]:
dataset.get_dictionary()

artm.Dictionary(name=b4e1ca21-0cb1-4e17-a695-fcd5663883a4, num_entries=892938)

In [16]:
dictionary = dataset.get_dictionary()

In [17]:
print(dictionary)

for modality in dataset.get_possible_modalities():
    if modality not in [MAIN_MODALITY]:
        dictionary.filter(class_id=modality, max_df=0, inplace=True)

artm.Dictionary(name=b4e1ca21-0cb1-4e17-a695-fcd5663883a4, num_entries=892938)


In [18]:
dictionary.filter(min_df=5, max_df_rate=0.5)

artm.Dictionary(name=b4e1ca21-0cb1-4e17-a695-fcd5663883a4, num_entries=61688)

In [19]:
dataset._cached_dict = dictionary

In [20]:
dataset.get_dictionary()

artm.Dictionary(name=b4e1ca21-0cb1-4e17-a695-fcd5663883a4, num_entries=61688)

In [21]:
%%time

occurences, co_occurences = calc_doc_occurrences(dataset, MAIN_MODALITY)

CPU times: user 53.6 s, sys: 1.49 s, total: 55.1 s
Wall time: 54.5 s


In [22]:
co_occurences.shape

(61688, 61688)

In [23]:
calc_pmi = create_pmi_top_function(
    occurences, co_occurences,
    dataset.get_dataset().shape[0], [20],
    topic_indices=[0, 1, 2],
    co_occurrences_smooth=1e-2,
)

In [24]:
class TopTokenCoherence(BaseTopicNetScore):
    def __init__(self, name, func):
        super().__init__()

        self._name = name
        self.calc_pmi = func

    def call(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[1]

    def call_by_topic(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[3]

In [26]:
KnownModel

<enum 'KnownModel'>

In [20]:
PARAMS_EXPLORED

{<KnownModel.LDA: 'LDA'>: {'prior': ['symmetric', 'asymmetric', 'heuristic']},
 <KnownModel.PLSA: 'PLSA'>: {},
 <KnownModel.TLESS: 'TARTM'>: {},
 <KnownModel.SPARSE: 'sparse'>: {'smooth_bcg_tau': [0.05, 0.1],
  'sparse_sp_tau': [-0.05, -0.1]},
 <KnownModel.DECORRELATION: 'decorrelation'>: {'decorrelation_tau': [0.02,
   0.05,
   0.1]},
 <KnownModel.ARTM: 'ARTM'>: {'smooth_bcg_tau': [0.05, 0.1],
  'sparse_sp_tau': [-0.05, -0.1],
  'decorrelation_tau': [0.02, 0.05, 0.1]}}

In [25]:
NUM_TOPICS = 50  # vary
NUM_TRAINS = 3
NUM_ITERATIONS = 10
NUM_TOP_TOKENS = 20

In [28]:
dataset.get_dictionary()

artm.Dictionary(name=2cb5b1ff-8de4-45ea-a340-81cbba32b39c, num_entries=61688)

In [29]:
dictionary = dataset.get_dictionary()

In [30]:
dictionary

artm.Dictionary(name=2cb5b1ff-8de4-45ea-a340-81cbba32b39c, num_entries=61688)

## Test

In [45]:
PARAMS_EXPLORED[KnownModel.PLSA]

{}

In [54]:
%%time

model = init_model_from_family(
    family=KnownModel.PLSA,
    dataset=dataset,
    main_modality=MAIN_MODALITY,
    num_topics=NUM_TOPICS,
    seed=1,
)

model._fit(dataset.get_batch_vectorizer(), num_iterations=10)

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


CPU times: user 3min 1s, sys: 2.88 s, total: 3min 4s
Wall time: 1min 16s


In [55]:
list(model.scores.keys())

['PerplexityScore@all',
 'SparsityThetaScore',
 'SparsityPhiScore@lemmatized',
 'PerplexityScore@lemmatized',
 'TopicKernel@lemmatized.average_coherence',
 'TopicKernel@lemmatized.average_contrast',
 'TopicKernel@lemmatized.average_purity',
 'TopicKernel@lemmatized.average_size',
 'TopicKernel@lemmatized.coherence',
 'TopicKernel@lemmatized.contrast',
 'TopicKernel@lemmatized.purity',
 'TopicKernel@lemmatized.size',
 'TopicKernel@lemmatized.tokens']

In [58]:
model.scores[f'PerplexityScore{MAIN_MODALITY}']

[60934.58203125,
 9283.0244140625,
 8148.603515625,
 6405.390625,
 5430.90283203125,
 4980.9423828125,
 4732.5595703125,
 4578.2822265625,
 4477.1103515625,
 4408.48046875]

In [59]:
phi = model.get_phi()
target_topic_indices = list(range(NUM_TOPICS))  # phi.columns.get_loc()
target_topic_names = [phi.columns[i] for i in target_topic_indices]

custom_scores = [
    TopTokenCoherence(
        name=f'coherence_{top}',
        func=create_pmi_top_function(
            occurences, co_occurences,
            dataset.get_dataset().shape[0], [top],
            topic_indices=target_topic_indices,
            co_occurrences_smooth=1e-2,
        )
    )
    for top in [20]  # [10, 20, 50, 100]
]
custom_scores = custom_scores + [
    DiversityScore(
        name=f'diversity_{metric}',
        topic_names=['topic_0', 'topic_1'],
        class_ids=MAIN_MODALITY,
    )

    for metric in KNOWN_METRICS
]

for score in custom_scores:
    res = score.call(model)

    print(score._name)
    print(res)

    if isinstance(score, TopTokenCoherence):
        res_by_topic = score.call_by_topic(model)

        print(res_by_topic)

coherence_20
[0.0273816]
{0: array([0.]), 1: array([0.02958778]), 2: array([0.04763114]), 3: array([0.01888187]), 4: array([0.]), 5: array([0.03611333]), 6: array([0.]), 7: array([0.09164026]), 8: array([0.04763114]), 9: array([0.]), 10: array([0.]), 11: array([0.07011646]), 12: array([0.]), 13: array([0.]), 14: array([0.]), 15: array([0.]), 16: array([0.]), 17: array([0.08207206]), 18: array([0.02869565]), 19: array([0.09526229])}
diversity_euclidean
0.038674582514403096
diversity_jensenshannon
0.038674582514403096
diversity_hellinger
0.038674582514403096
diversity_cosine
0.038674582514403096


In [60]:
model.get_phi(class_ids=MAIN_MODALITY)['topic_18'].sort_values(ascending=False)

modality     token          
@lemmatized  вид                0.011582
             птица              0.011062
             территория         0.006171
             район              0.005858
             река               0.005784
                                  ...   
             минималистичный    0.000000
             натурщица          0.000000
             афрасиябнуть       0.000000
             nadh               0.000000
             рлэ                0.000000
Name: topic_18, Length: 61688, dtype: float32

In [61]:
model.class_ids

{'@lemmatized': 1}

In [62]:
KNOWN_METRICS

['euclidean', 'jensenshannon', 'hellinger', 'cosine']

In [31]:
MAIN_MODALITY

'@lemmatized'

In [26]:
def fit_and_compute_scores(model, dataset):
    model._fit(dataset.get_batch_vectorizer(), num_iterations=NUM_ITERATIONS)

    score_values = {
        'perplexity': model.scores[f'PerplexityScore{MAIN_MODALITY}'][-1],
    }

    phi = model.get_phi()

    # Currently all topics are taken into account
    target_topic_indices = list(range(NUM_TOPICS))  # phi.columns.get_loc()
    target_topic_names = [phi.columns[i] for i in target_topic_indices]

    top = NUM_TOP_TOKENS
    coherence_score = TopTokenCoherence(
        name=f'coherence_{top}',
        func=create_pmi_top_function(
            occurences, co_occurences,
            dataset.get_dataset().shape[0], [top],
            topic_indices=target_topic_indices,
            co_occurrences_smooth=1e-2,
        )
    )

    value = coherence_score.call(model)
    score_values[coherence_score._name] = value
    topic_coherences = coherence_score.call_by_topic(model)
    topic_coherences = {t: float(v) for t, v in topic_coherences.items()}

    diversity_scores = [
        DiversityScore(
            name=f'diversity_{metric}',
            metric=metric,
            topic_names=target_topic_names,
            class_ids=MAIN_MODALITY,
        )
    
        for metric in KNOWN_METRICS
    ]
    
    for score in diversity_scores:
        value = score.call(model)
        score_values[score._name] = value

    return {
        'scores': score_values,
        'topic_coherences': topic_coherences,
    }

In [27]:
co_occurences.shape

(61688, 61688)

In [33]:
BEST_PARAMS = dict()

## PLSA

In [66]:
PARAMS_EXPLORED[KnownModel.PLSA]

{}

In [67]:
NUM_TOPICS

20

In [68]:
results = []

for seed in range(NUM_TRAINS):
    print(seed)
    
    model = init_model_from_family(
        family=KnownModel.PLSA,
        dataset=dataset,
        main_modality=MAIN_MODALITY,
        num_topics=NUM_TOPICS,
        seed=seed,
    )
    scores = fit_and_compute_scores(model, dataset)
    results.append(scores)

0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


In [34]:
BEST_PARAMS[KnownModel.PLSA] = None

## Sparse

In [76]:
PARAMS_EXPLORED[KnownModel.SPARSE]

{'smooth_bcg_tau': [0.05, 0.1], 'sparse_sp_tau': [-0.05, -0.1]}

In [71]:
results = dict()

for sparse_sp_tau in PARAMS_EXPLORED[KnownModel.SPARSE]['sparse_sp_tau']:
    for smooth_bcg_tau in PARAMS_EXPLORED[KnownModel.SPARSE]['smooth_bcg_tau']:
        key = (sparse_sp_tau, smooth_bcg_tau)
        results[key] = []

        print(key)

        for seed in range(NUM_TRAINS):
            print(seed)
            
            model = init_model_from_family(
                family=KnownModel.SPARSE,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={'sparse_sp_tau': sparse_sp_tau, 'smooth_bcg_tau': smooth_bcg_tau}
            )

            for reg in model.regularizers.data:
                print(f"{reg}: {model.regularizers[reg].tau}")
 
            scores = fit_and_compute_scores(model, dataset)
            results[key].append(scores)

        print()

    print()

(-0.05, 0.05)
0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.4713878133278989
sparse_theta_sp: -3.380096644027831
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.4713878133278989
sparse_theta_sp: -3.380096644027831
2


...

smooth_phi_bcg: 21.998097955301954
smooth_theta_bcg: 157.73784338796543
sparse_phi_sp: -0.8999221890805342
sparse_theta_sp: -6.452911774962223
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 21.998097955301954
smooth_theta_bcg: 157.73784338796543
sparse_phi_sp: -0.8999221890805342
sparse_theta_sp: -6.452911774962223
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 21.998097955301954
smooth_theta_bcg: 157.73784338796543
sparse_phi_sp: -0.8999221890805342
sparse_theta_sp: -6.452911774962223




In [77]:
results

{(-0.05,
  0.05): [{'scores': {'perplexity': 4855.087890625,
    'coherence_20': array([0.02440098]),
    'diversity_euclidean': 0.0514985041579102,
    'diversity_jensenshannon': 0.6619651830384224,
    'diversity_hellinger': 0.762892507105303,
    'diversity_cosine': 0.8218104318940633},
   'topic_coherences': {0: 0.0,
    1: 0.0,
    2: 0.09931197808327598,
    3: 0.047631142579751144,
    4: 0.0,
    5: 0.04641497529969045,
    6: 0.047631142579751144,
    7: 0.0,
    8: 0.0,
    9: 0.0,
    10: 0.02882142496497416,
    11: 0.03611332815466127,
    12: 0.0,
    13: 0.047631142579751144,
    14: 0.0,
    15: 0.0,
    16: 0.0,
    17: 0.09526228515950229,
    18: 0.0,
    19: 0.03920227108373217}}, {'scores': {'perplexity': 4924.4453125,
    'coherence_20': array([0.0330981]),
    'diversity_euclidean': 0.05188991960405022,
    'diversity_jensenshannon': 0.6644929983275759,
    'diversity_hellinger': 0.7671443148908221,
    'diversity_cosine': 0.8209736198511112},
   'topic_coherence

In [78]:
for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    print(k, mean_ppl)

(-0.05, 0.05) 4869.897135416667
(-0.05, 0.1) 5094.970865885417
(-0.1, 0.05) 5037.889485677083
(-0.1, 0.1) 5275.473795572917


In [79]:
# Best: (-0.05, 0.05) 4869.897135416667

In [35]:
BEST_PARAMS[KnownModel.SPARSE] = {
    'sparse_sp_tau': -0.05,
    'smooth_bcg_tau': 0.05,
}

## Decorrelation

In [92]:
PARAMS_EXPLORED[KnownModel.DECORRELATION]

{'decorrelation_tau': [0.02, 0.05, 0.1]}

In [85]:
PARAMS_EXPLORED[KnownModel.ARTM]

{'smooth_bcg_tau': [0.05, 0.1],
 'sparse_sp_tau': [-0.05, -0.1],
 'decorrelation_tau': [0.02, 0.05, 0.1]}

In [86]:
DECORRELATION_TAUS = [0.01] + PARAMS_EXPLORED[KnownModel.DECORRELATION]['decorrelation_tau']

In [87]:
results = dict()

for decorrelation_tau in DECORRELATION_TAUS:
    for smooth_bcg_tau in PARAMS_EXPLORED[KnownModel.SPARSE]['smooth_bcg_tau']:
        key = (decorrelation_tau, smooth_bcg_tau)
        results[key] = []

        print(key)

        for seed in range(NUM_TRAINS):
            print(seed)
            
            model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': decorrelation_tau,
                    'smooth_bcg_tau': smooth_bcg_tau,
                    'sparse_sp_tau': 0.0,
                }
            )

            for reg in model.regularizers.data:
                print(f"{reg}: {model.regularizers[reg].tau}")

            scores = fit_and_compute_scores(model, dataset)
            results[key].append(scores)

        print()

    print()

(0.01, 0.05)
0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.01
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.01
2


...

smooth_phi_bcg: 21.998097955301954
smooth_theta_bcg: 157.73784338796543
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.1
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 21.998097955301954
smooth_theta_bcg: 157.73784338796543
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.1
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 21.998097955301954
smooth_theta_bcg: 157.73784338796543
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.1




In [88]:
len(results)

8

In [89]:
results

{(0.01,
  0.05): [{'scores': {'perplexity': 4552.7373046875,
    'coherence_20': array([0.033042]),
    'diversity_euclidean': 0.04783356996692522,
    'diversity_jensenshannon': 0.6437350411279585,
    'diversity_hellinger': 0.7322378097442451,
    'diversity_cosine': 0.8180261060571838},
   'topic_coherences': {0: 0.047631142579751144,
    1: 0.0,
    2: 0.09931197808327598,
    3: 0.04763114257975122,
    4: 0.01888186700742297,
    5: 0.05746003791267812,
    6: 0.047631142579751144,
    7: 0.0,
    8: 0.0,
    9: 0.047631142579751144,
    10: 0.028821424964974234,
    11: 0.03611332815466127,
    12: 0.0,
    13: 0.047631142579751075,
    14: 0.0,
    15: 0.0,
    16: 0.0,
    17: 0.14289342773925337,
    18: 0.0,
    19: 0.03920227108373217}}, {'scores': {'perplexity': 4593.36181640625,
    'coherence_20': array([0.02011038]),
    'diversity_euclidean': 0.04646714677957182,
    'diversity_jensenshannon': 0.6461533679590886,
    'diversity_hellinger': 0.735953378971041,
    'diver

In [90]:
for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)

    print(k, mean_ppl)

(0.01, 0.05) 4564.923990885417
(0.01, 0.1) 4765.987955729167
(0.02, 0.05) 4564.5703125
(0.02, 0.1) 4766.520670572917
(0.05, 0.05) 4596.274088541667
(0.05, 0.1) 4791.552083333333
(0.1, 0.05) 4711.102376302083
(0.1, 0.1) 4888.488606770833


In [ ]:
#  Best: (0.02, 0.05) 4564.5703125
# Close: (0.01, 0.05) 4564.923990885417
#        (0.05, 0.05) 4596.274088541667

In [36]:
BEST_PARAMS[KnownModel.DECORRELATION] = {
    'decorrelation_tau': 0.02,
    'smooth_bcg_tau': 0.05,
}

## ARTM

In [94]:
PARAMS_EXPLORED[KnownModel.SPARSE]

{'smooth_bcg_tau': [0.05, 0.1], 'sparse_sp_tau': [-0.05, -0.1]}

In [95]:
PARAMS_EXPLORED[KnownModel.DECORRELATION]

{'decorrelation_tau': [0.02, 0.05, 0.1]}

In [96]:
PARAMS_EXPLORED[KnownModel.ARTM]

{'smooth_bcg_tau': [0.05, 0.1],
 'sparse_sp_tau': [-0.05, -0.1],
 'decorrelation_tau': [0.02, 0.05, 0.1]}

In [97]:
results = dict()

for decorrelation_tau in DECORRELATION_TAUS:
    for sparse_sp_tau in PARAMS_EXPLORED[KnownModel.ARTM]['sparse_sp_tau']:
        for smooth_bcg_tau in PARAMS_EXPLORED[KnownModel.SPARSE]['smooth_bcg_tau']:
            key = (decorrelation_tau, sparse_sp_tau, smooth_bcg_tau)
            results[key] = []
    
            print(key)
    
            for seed in range(NUM_TRAINS):
                print(seed)
                
                model = init_model_from_family(
                    family=KnownModel.ARTM,
                    dataset=dataset,
                    main_modality=MAIN_MODALITY,
                    num_topics=NUM_TOPICS,
                    seed=seed,
                    model_params={
                        'decorrelation_tau': decorrelation_tau,
                        'smooth_bcg_tau': smooth_bcg_tau,
                        'sparse_sp_tau': sparse_sp_tau,
                    }
                )
    
                for reg in model.regularizers.data:
                    print(f"{reg}: {model.regularizers[reg].tau}")
    
                scores = fit_and_compute_scores(model, dataset)
                results[key].append(scores)

            print()

        print()

    print()

(0.01, -0.05, 0.05)
0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.4713878133278989
sparse_theta_sp: -3.380096644027831
decorrelation: 0.01
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.4713878133278989
sparse_theta_sp: -3.380096644027831
decorrelation: 0.01
2


...

smooth_phi_bcg: 21.998097955301954
smooth_theta_bcg: 157.73784338796543
sparse_phi_sp: -0.8999221890805342
sparse_theta_sp: -6.452911774962223
decorrelation: 0.1
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 21.998097955301954
smooth_theta_bcg: 157.73784338796543
sparse_phi_sp: -0.8999221890805342
sparse_theta_sp: -6.452911774962223
decorrelation: 0.1
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 21.998097955301954
smooth_theta_bcg: 157.73784338796543
sparse_phi_sp: -0.8999221890805342
sparse_theta_sp: -6.452911774962223
decorrelation: 0.1





In [101]:
len(results)

16

In [102]:
ppls = []

for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    ppls.append(mean_ppl)

    print(k, mean_ppl)

(0.01, -0.05, 0.05) 4876.149739583333
(0.01, -0.05, 0.1) 5099.948079427083
(0.01, -0.1, 0.05) 5043.05615234375
(0.01, -0.1, 0.1) 5280.022298177083
(0.02, -0.05, 0.05) 4883.79248046875
(0.02, -0.05, 0.1) 5105.9921875
(0.02, -0.1, 0.05) 5049.108072916667
(0.02, -0.1, 0.1) 5285.413736979167
(0.05, -0.05, 0.05) 4915.909342447917
(0.05, -0.05, 0.1) 5132.880696614583
(0.05, -0.1, 0.05) 5075.96142578125
(0.05, -0.1, 0.1) 5312.384765625
(0.1, -0.05, 0.05) 4995.532389322917
(0.1, -0.05, 0.1) 5205.424153645833
(0.1, -0.1, 0.05) 5156.685709635417
(0.1, -0.1, 0.1) 5383.19482421875


In [103]:
sorted(ppls)[:5]

[4876.149739583333,
 4883.79248046875,
 4915.909342447917,
 4995.532389322917,
 5043.05615234375]

In [ ]:
#  Best: (0.01, -0.05, 0.05) 4876.149739583333
# Close: (0.02, -0.05, 0.05) 4883.79248046875

In [37]:
BEST_PARAMS[KnownModel.DECORRELATION] = {
    'decorrelation_tau': 0.01,
    'sparse_sp_tau':    -0.05,
    'smooth_bcg_tau':    0.05,
}

## TLESS

In [38]:
PARAMS_EXPLORED[KnownModel.TLESS]

{}

In [106]:
results = []

for seed in range(NUM_TRAINS):
    print(seed)
    
    model = init_model_from_family(
        family=KnownModel.TLESS,
        dataset=dataset,
        main_modality=MAIN_MODALITY,
        num_topics=NUM_TOPICS,
        seed=seed,
    )
    scores = fit_and_compute_scores(model, dataset)
    results.append(scores)

0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


In [107]:
results

[{'scores': {'perplexity': 5746.67724609375,
   'coherence_20': array([0.02294889]),
   'diversity_euclidean': 0.06061311426052078,
   'diversity_jensenshannon': 0.7158020409439757,
   'diversity_hellinger': 0.8247900878167469,
   'diversity_cosine': 0.8924229196266671},
  'topic_coherences': {0: 0.0,
   1: 0.0,
   2: 0.04597036819475651,
   3: 0.047631142579751075,
   4: 0.0,
   5: 0.04400911946807843,
   6: 0.047631142579751144,
   7: 0.0,
   8: 0.047631142579751144,
   9: 0.0,
   10: 0.047631142579751144,
   11: 0.0,
   12: 0.0,
   13: 0.09526228515950215,
   14: 0.04400911946807843,
   15: 0.0,
   16: 0.0,
   17: 0.0,
   18: 0.0,
   19: 0.03920227108373217}},
 {'scores': {'perplexity': 5784.98974609375,
   'coherence_20': array([0.02947549]),
   'diversity_euclidean': 0.06056725134639164,
   'diversity_jensenshannon': 0.7175657037435673,
   'diversity_hellinger': 0.8272357997860749,
   'diversity_cosine': 0.8930457195406656},
  'topic_coherences': {0: 0.0,
   1: 0.02593965789275605

In [108]:
# Best:

In [39]:
BEST_PARAMS[KnownModel.TLESS] = None

## LDA

In [110]:
PARAMS_EXPLORED[KnownModel.LDA]

{'prior': ['symmetric', 'asymmetric', 'heuristic']}

In [111]:
results = dict()

for prior in PARAMS_EXPLORED[KnownModel.LDA]['prior']:
    key = prior
    results[key] = []

    print(key)

    for seed in range(NUM_TRAINS):
        print(seed)
        
        model = init_model_from_family(
            family=KnownModel.LDA,
            dataset=dataset,
            main_modality=MAIN_MODALITY,
            num_topics=NUM_TOPICS,
            seed=seed,
            model_params={'prior': prior}
        )

        for reg in model.regularizers.data:
            print(f"{reg}: {model.regularizers[reg].tau}")

        scores = fit_and_compute_scores(model, dataset)
        results[key].append(scores)

    print()

symmetric
0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi: 0.05
smooth_theta: 0.05
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi: 0.05
smooth_theta: 0.05
2


...

smooth_phi: 0.01
smooth_theta: 2.5
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi: 0.01
smooth_theta: 2.5
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi: 0.01
smooth_theta: 2.5



In [112]:
results

{'symmetric': [{'scores': {'perplexity': 4423.35400390625,
    'coherence_20': array([0.02790284]),
    'diversity_euclidean': 0.042772612034107746,
    'diversity_jensenshannon': 0.6213137918726418,
    'diversity_hellinger': 0.6951459636572607,
    'diversity_cosine': 0.7893040436876423},
   'topic_coherences': {0: 0.0,
    1: 0.0,
    2: 0.09360151077450765,
    3: 0.04763114257975122,
    4: 0.0,
    5: 0.035369912686702774,
    6: 0.047631142579751144,
    7: 0.0,
    8: 0.0,
    9: 0.0,
    10: 0.0,
    11: 0.03611332815466127,
    12: 0.0,
    13: 0.047631142579751144,
    14: 0.0,
    15: 0.0,
    16: 0.047631142579751075,
    17: 0.16324529624506007,
    18: 0.0,
    19: 0.03920227108373217}},
  {'scores': {'perplexity': 4414.8154296875,
    'coherence_20': array([0.0273816]),
    'diversity_euclidean': 0.04318211650310182,
    'diversity_jensenshannon': 0.6243635399351967,
    'diversity_hellinger': 0.6992751663679955,
    'diversity_cosine': 0.7945430170962635},
   'topic_co

In [113]:
ppls = []

for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    ppls.append(mean_ppl)

    print(k, mean_ppl)

symmetric 4408.221842447917
asymmetric 4405.46337890625
heuristic 4508.47509765625


In [114]:
sorted(ppls)

[4405.46337890625, 4408.221842447917, 4508.47509765625]

In [116]:
# Best: symmetric 4408.221842447917

In [40]:
BEST_PARAMS[KnownModel.LDA] = {
    'prior': 'symmetric',
}

In [118]:
BEST_PARAMS

{<KnownModel.PLSA: 'PLSA'>: None,
 <KnownModel.SPARSE: 'sparse'>: {'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.DECORRELATION: 'decorrelation'>: {'decorrelation_tau': 0.01,
  'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.TLESS: 'TARTM'>: None,
 <KnownModel.LDA: 'LDA'>: {'prior': 'symmetric'}}

In [28]:
BEST_PARAMS = {KnownModel.PLSA: None,
 KnownModel.SPARSE: {'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 KnownModel.DECORRELATION: {'decorrelation_tau': 0.01,
  'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 KnownModel.TLESS: None,
 KnownModel.LDA: {'prior': 'symmetric'}}

In [29]:
import json
import warnings

warnings.simplefilter('ignore', UserWarning)

In [30]:
NUM_TRAINS = 20  # 100
COHERENCES = list()

In [31]:
! ls results

20newsgroups  mkb10  postnauka	rtlwikiperson  ruwikigood


In [32]:
SAVE_FOLDER = 'results50/ruwikigood'

! mkdir -p $SAVE_FOLDER

In [123]:
! ls

20_Newsgroups__internals		phi1.batch
ARTM-Models-20NewsGroups-T20.ipynb	Post_Science__internals
ARTM-Models-PostNauka-T20.ipynb		results
ARTM-Models-RuWikiGood-T20.ipynb	TestTopicNet.ipynb
BERTopic				TopicBank-20NewsGroups-T20.ipynb
Experiment_v2.ipynb			TopicBank-PostNauka-T20-Copy1.ipynb
Experiment_v3.ipynb			TopicBank-PostNauka-T20.ipynb
Iterative-Model-20NewsGroups-T20.ipynb	Topic-Thetaless-Regularizer.ipynb
Iterative-Model-PostNauka-T20.ipynb


In [33]:
# PLSA

scores = []

# for seed in tqdm(range(NUM_TRAINS), total=NUM_TRAINS):
for seed in range(NUM_TRAINS):
    if seed != NUM_TRAINS - 1:
        print(seed, end=' ')
    else:
        print(seed)

    model = init_model_from_family(
        family=KnownModel.PLSA,
        dataset=dataset,
        main_modality=MAIN_MODALITY,
        num_topics=NUM_TOPICS,
        seed=seed,
    )
    results = fit_and_compute_scores(model, dataset)
    # scores.append(results['scores'])
    scores.append(results)

    COHERENCES.extend(
        list(results['topic_coherences'].values())
    )

0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19


In [34]:
scores[0]

{'scores': {'perplexity': 3507.619140625,
  'coherence_20': array([0.79623148]),
  'diversity_euclidean': 0.056667253871222095,
  'diversity_jensenshannon': 0.6595447920784969,
  'diversity_hellinger': 0.7535439018530243,
  'diversity_cosine': 0.8561366596800426},
 'topic_coherences': {0: 0.5503435038712955,
  1: 0.5228143913524883,
  2: 0.5223335709100484,
  3: 0.7304031766123252,
  4: 0.9740404043527446,
  5: 0.709119920513533,
  6: 0.8953457254677054,
  7: 0.521520021630769,
  8: 0.45287697661575343,
  9: 1.504778128596866,
  10: 0.8733053154552386,
  11: 1.1339720822955437,
  12: 0.9099572743337381,
  13: 0.7819548896543764,
  14: 0.8042061667795852,
  15: 0.6465235066393118,
  16: 0.5562163963700724,
  17: 0.5764732213051018,
  18: 0.9187080477478147,
  19: 0.600260825738937,
  20: 1.1772452991715994,
  21: 0.6399363211274745,
  22: 0.7573471153724096,
  23: 0.9065397831442783,
  24: 0.5668064703314608,
  25: 1.0521742682269883,
  26: 0.76411246828742,
  27: 1.1482536314762812,
  

In [35]:
# for s in scores:
#     s['coherence_20'] = float(s['coherence_20'])

for s in scores:
    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

In [48]:
scores

[{'perplexity': 3507.619140625,
  'coherence_20': 0.7962314838647656,
  'diversity_euclidean': 0.056667254394286534,
  'diversity_jensenshannon': 0.6595447894329457,
  'diversity_hellinger': 0.7535439037130035,
  'diversity_cosine': 0.8561366559298873},
 {'perplexity': 3511.848876953125,
  'coherence_20': 0.7948478920615168,
  'diversity_euclidean': 0.05524234540362837,
  'diversity_jensenshannon': 0.6553867953987579,
  'diversity_hellinger': 0.7477917725409869,
  'diversity_cosine': 0.849887948142836},
 {'perplexity': 3473.005859375,
  'coherence_20': 0.8348717636324882,
  'diversity_euclidean': 0.05740241789034991,
  'diversity_jensenshannon': 0.6576794456144781,
  'diversity_hellinger': 0.7507478518669871,
  'diversity_cosine': 0.8586577549074998},
 {'perplexity': 3442.25244140625,
  'coherence_20': 0.8487344248362076,
  'diversity_euclidean': 0.05613743850601633,
  'diversity_jensenshannon': 0.6582932742045129,
  'diversity_hellinger': 0.7522375860591992,
  'diversity_cosine': 0.85

In [49]:
SAVE_FOLDER

'results50/ruwikigood'

In [36]:
with open(SAVE_FOLDER + '/plsa_with_cohs.json', 'w') as f:
    f.write(
        json.dumps(scores, indent=4)
    )

In [51]:
len(COHERENCES)

1000

In [52]:
COHERENCES[:10]

[0.5503435038712955,
 0.5228143913524883,
 0.5223335709100484,
 0.7304031766123252,
 0.9740404043527446,
 0.709119920513533,
 0.8953457254677054,
 0.521520021630769,
 0.45287697661575343,
 1.504778128596866]

In [53]:
COHERENCES[:20]

[0.5503435038712955,
 0.5228143913524883,
 0.5223335709100484,
 0.7304031766123252,
 0.9740404043527446,
 0.709119920513533,
 0.8953457254677054,
 0.521520021630769,
 0.45287697661575343,
 1.504778128596866,
 0.8733053154552386,
 1.1339720822955437,
 0.9099572743337381,
 0.7819548896543764,
 0.8042061667795852,
 0.6465235066393118,
 0.5562163963700724,
 0.5764732213051018,
 0.9187080477478147,
 0.600260825738937]

In [37]:
# Sparse

scores = []

# for seed in tqdm(range(NUM_TRAINS), total=NUM_TRAINS):
for seed in range(NUM_TRAINS):
    if seed != NUM_TRAINS - 1:
        print(seed, end=' ')
    else:
        print(seed)

    model = init_model_from_family(
        family=KnownModel.SPARSE,
        dataset=dataset,
        main_modality=MAIN_MODALITY,
        num_topics=NUM_TOPICS,
        seed=seed,
        model_params=BEST_PARAMS[KnownModel.SPARSE],
    )

    if seed == 0:
        for reg in model.regularizers.data:
            print(f"{reg}: {model.regularizers[reg].tau}")

    results = fit_and_compute_scores(model, dataset)
    # scores.append(results['scores'])
    scores.append(results)

    COHERENCES.extend(
        list(results['topic_coherences'].values())
    )

0 smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.1885551253311596
sparse_theta_sp: -1.3520386576111325
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19


In [38]:
# for s in scores:
#     s['coherence_20'] = float(s['coherence_20'])

for s in scores:
    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

In [39]:
with open(SAVE_FOLDER + '/sparse_with_cohs.json', 'w') as f:
    f.write(
        json.dumps(scores, indent=4)
    )

In [57]:
len(COHERENCES)

2000

In [58]:
COHERENCES[-10:]

[1.1150787266720121,
 1.3409300601643854,
 0.748489761227004,
 0.9818319065906047,
 0.8016536596658803,
 0.8911041065733367,
 1.1463807982854117,
 0.6663806173485558,
 0.7303251608196153,
 0.5981704617548399]

In [59]:
max(COHERENCES)

2.145217958975005

In [60]:
min(COHERENCES)

0.37469969357017097

In [40]:
def train_many(model_family, save_file_path):
    scores = []

    # for seed in tqdm(range(NUM_TRAINS), total=NUM_TRAINS):
    for seed in range(NUM_TRAINS):
        if seed != NUM_TRAINS - 1:
            print(seed, end=' ')
        else:
            print(seed)
    
        model = init_model_from_family(
            family=model_family,
            dataset=dataset,
            main_modality=MAIN_MODALITY,
            num_topics=NUM_TOPICS,
            seed=seed,
            model_params=BEST_PARAMS[model_family],
        )
    
        if seed == 0:
            for reg in model.regularizers.data:
                print(f"{reg}: {model.regularizers[reg].tau}")
    
        results = fit_and_compute_scores(model, dataset)
        # scores.append(results['scores'])
        scores.append(results)
    
        COHERENCES.extend(
            list(results['topic_coherences'].values())
        )

    # for s in scores:
    #     s['coherence_20'] = float(s['coherence_20'])
    
    for s in scores:
        s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

    with open(save_file_path, 'w') as f:
        f.write(
            json.dumps(scores, indent=4)
        )

In [41]:
train_many(KnownModel.DECORRELATION, SAVE_FOLDER + '/decorrelation_with_cohs.json')

0 decorrelation: 0.01
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19


In [63]:
len(COHERENCES)

3000

In [42]:
train_many(KnownModel.TLESS, SAVE_FOLDER + '/tless_with_cohs.json')

0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19


In [65]:
len(COHERENCES)

4000

In [43]:
train_many(KnownModel.LDA, SAVE_FOLDER + '/lda_with_cohs.json')

0 smooth_phi: 0.02
smooth_theta: 0.02
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19


In [45]:
len(COHERENCES)

5000

In [68]:
COHERENCES

[0.5503435038712955,
 0.5228143913524883,
 0.5223335709100484,
 0.7304031766123252,
 0.9740404043527446,
 0.709119920513533,
 0.8953457254677054,
 0.521520021630769,
 0.45287697661575343,
 1.504778128596866,
 0.8733053154552386,
 1.1339720822955437,
 0.9099572743337381,
 0.7819548896543764,
 0.8042061667795852,
 0.6465235066393118,
 0.5562163963700724,
 0.5764732213051018,
 0.9187080477478147,
 0.600260825738937,
 1.1772452991715994,
 0.6399363211274745,
 0.7573471153724096,
 0.9065397831442783,
 0.5668064703314608,
 1.0521742682269883,
 0.76411246828742,
 1.1482536314762812,
 0.8497681132329262,
 0.9978829035105352,
 1.3007583215341572,
 0.854283107978733,
 0.6710235835957812,
 0.9736974217033453,
 0.7444862487339565,
 0.8193002246031341,
 0.9458989769331125,
 1.2242458943357069,
 1.1150787266720121,
 0.7230680439356225,
 0.5027103442230393,
 0.7502385602100483,
 0.5843153155931237,
 0.5424995375123758,
 0.793913892422964,
 0.6905228270784843,
 0.5385696458655739,
 0.6031781804033832,

In [69]:
for p in range(5, 100, 5):
    print(f'{p:2}: {np.percentile(COHERENCES, p)}')

 5: 0.4348854355715953
10: 0.4932563028424238
15: 0.5372745322944787
20: 0.5677332839669632
25: 0.6022386177325694
30: 0.6361246805165369
35: 0.6748252222297236
40: 0.709056840872004
45: 0.7413237368660855
50: 0.772774211924556
55: 0.8033478822601885
60: 0.8355799150040666
65: 0.8749597821518427
70: 0.9128652809370966
75: 0.9514511005954257
80: 0.9918260892662304
85: 1.052162915274115
90: 1.1286470106781974
95: 1.2815636271035824


In [73]:
min(COHERENCES), max(COHERENCES)

(0.1853728816542277, 2.145217958975005)

In [74]:
np.argmin(COHERENCES), np.argmax(COHERENCES)

(2199, 1493)

In [75]:
for p in [2, 98]:
    print(f'{p:2}: {np.percentile(COHERENCES, p)}')

 2: 0.3678987873093018
98: 1.4813228091288768


In [218]:
BEST_PARAMS

{<KnownModel.PLSA: 'PLSA'>: None,
 <KnownModel.SPARSE: 'sparse'>: {'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.DECORRELATION: 'decorrelation'>: {'decorrelation_tau': 0.01,
  'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.TLESS: 'TARTM'>: None,
 <KnownModel.LDA: 'LDA'>: {'prior': 'symmetric'}}

In [222]:
dataset.get_dictionary()

artm.Dictionary(name=023f347a-db8f-45a1-ae63-2fc20bb7a634, num_entries=61688)

In [221]:
model.get_phi()['topic_10'].sort_values(ascending=False)[:20]

modality     token       
@lemmatized  клетка          0.010790
             вид             0.008763
             акула           0.006019
             ген             0.004679
             белок           0.004298
             форма           0.004195
             группа          0.004110
             белка           0.003864
             рнк             0.003845
             днк             0.003691
             случай          0.003654
             тип             0.003619
             система         0.003560
             гэс             0.003501
             тело            0.003409
             исследование    0.003405
             вирус           0.003364
             происходить     0.003343
             различный       0.003282
             развитие        0.003262
Name: topic_10, dtype: float32